## Import CSV's and Python Libraries

Run `pip install -r requirements.txt` in your terminal to install the packages necessary to run this notebook on your local machine.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("csv's/tm_ktx_merged.csv")
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
print(f"\nPitch types:\n{df['pitch_type'].value_counts()}")
print(f"\nUnique pitchers: {df['pitcher_name'].nunique()}")

Loaded 20,291 rows × 382 columns

Pitch types:
pitch_type
FB    8104
CH    3447
SL    2976
SI    2620
CB    1998
CT     651
SP     355
SW      89
KN      19
Name: count, dtype: int64

Unique pitchers: 965


## Clean Data

Removes warmup pitches and rows missing core pitch metrics, standardizes pitch type labels, and normalizes all left-handed pitcher metrics to a right-handed convention (flipping horizontal-axis values) so every downstream comparison is apples-to-apples.

In [ ]:
# Pitch type label → standard abbreviation
# Covers both full words and the short abbreviations this dataset actually uses
PITCH_TYPE_MAP = {
    # Full-word variants
    'Fastball':          'FF',
    'Four-Seam':         'FF',
    'FourSeamFastBall':  'FF',
    'Sinker':            'SI',
    'TwoSeamFastBall':   'SI',
    'Two-Seam':          'SI',
    'Slider':            'SL',
    'Sweeper':           'SW',
    'Curveball':         'CU',
    'Changeup':          'CH',
    'ChangeUp':          'CH',
    'Cutter':            'FC',
    'Splitter':          'FS',
    'Knuckleball':       'KN',
    # Short abbreviations present in this dataset
    'FB':                'FF',
    'CB':                'CU',
    'CT':                'FC',
    'SP':                'FS',
    'SW':                'SW',
}

# Columns required for a valid pitch record
REQUIRED_COLS = ['pitch_type', 'pitcher_throws', 'release_speed', 'spin_rate',
                 'induced_vertical_break', 'horizontal_break']

# Horizontal-axis columns that must be sign-flipped for LHP
HORIZONTAL_COLS = ['horizontal_break', 'release_side',
                   'plate_location_side', 'horizontal_approach_angle']

df_clean = df.copy()

# Drop warmup pitches
if 'is_warmup' in df_clean.columns:
    df_clean = df_clean[df_clean['is_warmup'] != 1]

# Drop rows missing essential pitch data
df_clean = df_clean.dropna(subset=REQUIRED_COLS).copy()

# Standardize pitch type labels
df_clean['pitch_type_std'] = (
    df_clean['pitch_type']
    .map(PITCH_TYPE_MAP)
    .fillna(df_clean['pitch_type'])
)


# Normalize LHP → RHP convention (flip all horizontal metrics)print(f"\nHandedness distribution:\n{df_clean['pitcher_throws'].value_counts()}")

lhp_mask = df_clean['pitcher_throws'].str.upper().str.startswith('L')
print(f"\nPitch type distribution:\n{df_clean['pitch_type_std'].value_counts()}")

for col in HORIZONTAL_COLS:print(f"Rows after cleaning: {len(df_clean):,}"):
    if col in df_clean.columns:
        df_clean.loc[lhp_mask, col] = -df_clean.loc[lhp_mask, col]

IndentationError: unexpected indent (752316157.py, line 59)

## Stuff+ Model

Stuff+ is computed per pitch type using a **weighted z-score composite** of key pitch characteristics. Each feature is z-scored within its pitch type group so that the units are consistent, then multiplied by a directional weight that encodes domain knowledge (e.g. more induced vertical break is *good* for a four-seamer, while more drop — negative IVB — is *good* for a curveball).

The composite is then rescaled so that **100 = average and σ = 10**, matching the conventional Stuff+ scale.

| Pitch | Key drivers |
|-------|-------------|
| FF | Velocity · IVB · Spin rate · Extension |
| SI | Velocity · Arm-side run · Sink (−IVB) |
| SL | Velocity · Glove-side break · Spin rate · VAA |
| CU | Spin rate · Drop (−IVB) · Velocity · VAA |
| CH | Arm-side fade · Arm-side drop · Low spin · Velocity differential |
| FC | Velocity · Glove-side cut · Spin rate |
| FS | Drop (−IVB) · Velocity · Arm-side run |

In [ ]:
# ---------------------------------------------------------------------------
# Feature weights per pitch type
# Positive weight  → more of this metric is better stuff
# Negative weight  → less of this metric is better stuff
# All horizontal directions assume RHP convention (LHP already flipped above)
# ---------------------------------------------------------------------------
STUFF_WEIGHTS = {
    'FF': {  # Four-Seam Fastball
        'release_speed':           0.45,   # raw velocity
        'induced_vertical_break':  0.30,   # ride / backspin carry
        'spin_rate':               0.15,   # backspin
        'extension':               0.10,   # perceived velocity gain
    },
    'SI': {  # Sinker / Two-Seam
        'release_speed':           0.40,
        'horizontal_break':        0.25,   # arm-side run (positive = arm-side for RHP)
        'induced_vertical_break': -0.25,   # sink (more negative IVB = more drop = better)
        'extension':               0.10,
    },
    'SL': {  # Slider
        'release_speed':           0.30,
        'horizontal_break':       -0.30,   # glove-side break (negative = glove-side for RHP)
        'spin_rate':               0.20,
        'vertical_approach_angle':-0.20,   # steeper descent = better (more negative VAA)
    },
    'CU': {  # Curveball
        'spin_rate':               0.30,
        'induced_vertical_break': -0.35,   # 12-6 drop (more negative IVB = better)
        'release_speed':           0.20,
        'vertical_approach_angle':-0.15,   # steeper = better
    },
    'CH': {  # Changeup
        'horizontal_break':        0.35,   # arm-side fade
        'induced_vertical_break':  0.25,   # tumbling action (positive IVB)
        'spin_rate':              -0.25,   # low spin = more tumble / deception
        'release_speed':          -0.15,   # velocity differential (lower relative = better)
    },
    'FC': {  # Cutter
        'release_speed':           0.40,
        'horizontal_break':       -0.35,   # glove-side cut
        'spin_rate':               0.15,
        'induced_vertical_break':  0.10,
    },
    'FS': {  # Splitter
        'induced_vertical_break': -0.45,   # late diving action
        'release_speed':           0.30,
        'horizontal_break':        0.15,   # arm-side
        'spin_rate':              -0.10,   # lower spin = more dive
    },
    'SW': {  # Sweeper
        'horizontal_break':       -0.45,   # glove-side sweep is the defining feature
        'release_speed':           0.25,
        'spin_rate':               0.20,
        'vertical_approach_angle':-0.10,   # some depth helps
    },
}

# Fallback for unlisted pitch types
DEFAULT_WEIGHTS = {
    'release_speed':           0.50,
    'spin_rate':               0.25,
    'induced_vertical_break':  0.25,
}


def _weighted_composite(group_df: pd.DataFrame, weights: dict) -> pd.Series:
    """
    For each feature in `weights`, z-score within `group_df`, multiply by its
    directional weight, then sum and normalise by total absolute weight.
    Returns a Series aligned to group_df.index.
    """
    composite = pd.Series(0.0, index=group_df.index)
    total_abs_weight = 0.0

    for feature, weight in weights.items():
        if feature not in group_df.columns:
            continue
        col = group_df[feature].dropna()
        if len(col) < 10 or col.std() == 0:
            continue
        z = (group_df[feature] - col.mean()) / col.std()
        composite += weight * z
        total_abs_weight += abs(weight)

    if total_abs_weight > 0:
        composite /= total_abs_weight
    return composite


# --- Step 1: raw composite score per pitch type ---
df_clean['_stuff_raw'] = np.nan

for pt, grp in df_clean.groupby('pitch_type_std'):
    weights = STUFF_WEIGHTS.get(pt, DEFAULT_WEIGHTS)
    df_clean.loc[grp.index, '_stuff_raw'] = _weighted_composite(grp, weights)

# --- Step 2: scale to 100 = average, σ = 10 (within each pitch type) ---
df_clean['stuff_plus'] = np.nan

for pt, grp in df_clean.groupby('pitch_type_std'):
    raw = df_clean.loc[grp.index, '_stuff_raw'].dropna()
    if len(raw) > 1 and raw.std() > 0:
        scaled = 100 + 10 * (raw - raw.mean()) / raw.std()
    else:
        scaled = pd.Series(100.0, index=raw.index)
    df_clean.loc[scaled.index, 'stuff_plus'] = scaled


print("Stuff+ distribution by pitch type:"))

print(    .round(1)

    df_clean.groupby('pitch_type_std')['stuff_plus']    .describe()[['count', 'mean', 'std', 'min', '50%', 'max']]

Stuff+ distribution by pitch type:
                 count   mean   std    min    50%    max
pitch_type_std                                          
CB              1630.0  100.0  10.0   65.2  100.7  124.7
CH              2908.0  100.0  10.0   39.1  100.0  140.5
CT               560.0  100.0  10.0   58.2  101.5  121.9
FB              7623.0  100.0  10.0   55.2  101.7  142.2
KN                 2.0  100.0   0.0  100.0  100.0  100.0
SI              2426.0  100.0  10.0   56.8  101.1  127.1
SL              2427.0  100.0  10.0   56.2  100.5  127.2
SP               274.0  100.0  10.0   74.0  100.5  127.5
SW                79.0  100.0  10.0   71.5  100.9  126.5


In [ ]:
# Per-pitcher, per-pitch-type summary
summary = (
    df_clean
    .groupby(['pitcher_name', 'pitch_type_std'], as_index=False)
    .agg(
        pitches    = ('stuff_plus',             'count'),
        stuff_plus = ('stuff_plus',             'mean'),
        velocity   = ('release_speed',          'mean'),
        spin_rate  = ('spin_rate',              'mean'),
        ivb        = ('induced_vertical_break', 'mean'),
        h_break    = ('horizontal_break',       'mean'),
        extension  = ('extension',              'mean'),
    )
    .round(1)
    .sort_values(['pitcher_name', 'stuff_plus'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 120)
display(summary)

,pitcher_name,pitch_type_std,pitches,stuff_plus,velocity,spin_rate,ivb,h_break,extension
0,Aaron Alston,CB,1,114.2,74.9,2310.0,3.7,-4.5,5.0
1,Aaron Alston,CH,3,109.4,76.8,1538.4,17.1,14.1,5.6
2,Aaron Alston,FB,4,100.7,83.3,1993.1,18.8,7.6,5.7
3,Aaron Alston,SI,4,88.1,81.1,1810.2,19.3,9.8,5.7
4,Aaron Alston,SL,2,87.4,76.1,1903.9,6.1,1.5,5.2
...,...,...,...,...,...,...,...,...,...
3662,Zander Street,CB,2,105.2,69.5,2148.2,1.6,-10.7,4.4
3663,Zander Street,CH,2,104.0,75.2,1843.4,8.5,19.5,5.0
3664,Zander Street,SI,2,99.2,81.5,1997.6,8.6,17.6,4.3
3665,Zander Street,FB,9,98.4,83.3,2091.1,14.5,11.0,5.0


## Pitcher Lookup

Set `pitcher_name` in the cell below to any pitcher's name exactly as it appears in the data, then run the cell to see every pitch they threw along with its Stuff+ score and key metrics.

In [ ]:
pitcher_name = "Eli Stewart"  # ← change this to the pitcher you want to look up

LOOKUP_COLS = [
    'date', 'pitch_type_std', 'release_speed', 'spin_rate',
    'induced_vertical_break', 'horizontal_break', 'vertical_approach_angle',
    'extension', 'stuff_plus',
]

cols_present = [c for c in LOOKUP_COLS if c in df_clean.columns]
pitcher_df = df_clean.loc[df_clean['pitcher_name'] == pitcher_name, cols_present].copy()

if pitcher_df.empty:
    print(f"No pitches found for '{pitcher_name}'.")
    print("Available pitchers (first 20):", df_clean['pitcher_name'].unique()[:20].tolist())
else:
    pitcher_df = pitcher_df.sort_values(['date', 'pitch_type_std']).reset_index(drop=True)
    pitcher_df = pitcher_df.round(1)
    print(f"{pitcher_name}  —  {len(pitcher_df)} pitches\n")
    display(pitcher_df)

Eli Stewart  —  28 pitches



,date,pitch_type_std,release_speed,spin_rate,induced_vertical_break,horizontal_break,vertical_approach_angle,extension,stuff_plus
0,2024-05-22,CB,78.4,2350.2,-7.5,-4.1,-6.5,4.1,112.4
1,2024-05-22,CB,79.4,2254.9,0.0,-2.8,-8.4,4.5,116.7
2,2024-05-22,CB,79.0,2373.6,-1.7,-4.4,-6.5,4.2,116.5
3,2024-05-22,CH,78.9,2359.2,13.0,20.8,-4.4,5.0,98.8
4,2024-05-22,CH,79.5,2286.1,5.9,23.8,-5.2,4.7,97.4
5,2024-05-22,CH,77.2,2365.4,10.3,21.7,-6.5,5.3,98.4
6,2024-05-22,FB,86.7,2376.2,15.0,19.5,-2.0,5.3,106.6
7,2024-05-22,FB,85.7,2518.7,17.1,17.3,-4.4,5.2,109.7
8,2024-05-22,FB,85.9,2514.5,18.0,20.0,-6.7,5.3,110.6
9,2024-05-22,FB,85.5,2554.7,15.5,19.6,-5.1,5.1,108.6


## Output Data

Saves two CSV files to the `csv's/` folder:
- **`stuff_plus_pitch_level.csv`** — one row per pitch with the computed Stuff+ score alongside core pitch metrics
- **`stuff_plus_summary.csv`** — one row per pitcher × pitch type with aggregated averages

In [ ]:
PITCH_LEVEL_COLS = [
    'pitcher_name', 'pitcher_throws', 'date', 'pitch_type_std',
    'release_speed', 'spin_rate', 'induced_vertical_break',
    'horizontal_break', 'vertical_approach_angle',
    'spin_efficiency', 'extension', 'release_height', 'release_side',
    'stuff_plus',
]

# Only keep columns that exist in the dataframe
pitch_level_cols_present = [c for c in PITCH_LEVEL_COLS if c in df_clean.columns]
out_pitch = df_clean[pitch_level_cols_present].copy()

out_pitch.to_csv("csv's/stuff_plus_pitch_level.csv", index=False)
summary.to_csv("csv's/stuff_plus_summary.csv", index=False)

print(f"Saved {len(out_pitch):,} pitch-level rows  → csv's/stuff_plus_pitch_level.csv")
print(f"Saved {len(summary):,} pitcher-summary rows → csv's/stuff_plus_summary.csv")

Saved 17,959 pitch-level rows  → csv's/stuff_plus_pitch_level.csv
Saved 3,667 pitcher-summary rows → csv's/stuff_plus_summary.csv


## Clean original CSV

Below is the code that was ran to transform "tm_ktx_merged.csv" to "tm_stuff_plus.csv". Both are present in the /csv's directory

In [ ]:
# Columns to keep from the original CSV — identifiers + every Stuff+ input
FOCUSED_COLS = [
    # --- Identifiers ---
    'pitcher_name',
    'pitcher_team',
    'pitcher_level',
    'pitcher_throws',
    'date',
    'pitch_type',
    'pitch_number',
    # --- Pitch characteristics used by the Stuff+ model ---
    'release_speed',
    'spin_rate',
    'induced_vertical_break',
    'horizontal_break',
    'vertical_approach_angle',
    'horizontal_approach_angle',
    'vertical_break',
    'spin_efficiency',
    'spin_axis',
    'tilt',
    'extension',
    'release_height',
    'release_side',
    'plate_location_height',
    'plate_location_side',
]

focused_present = [c for c in FOCUSED_COLS if c in df.columns]
df_focused = df[focused_present].copy()

df_focused.to_csv("csv's/tm_stuff_plus_focused.csv", index=False)
print(f"Saved {len(df_focused):,} rows × {len(focused_present)} columns → csv's/tm_stuff_plus_focused.csv")
print(f"Columns dropped: {df.shape[1] - len(focused_present)} of {df.shape[1]}")
print(f"Columns kept: {focused_present}")

Saved 20,291 rows × 22 columns → csv's/tm_stuff_plus_focused.csv
Columns dropped: 360 of 382
Columns kept: ['pitcher_name', 'pitcher_team', 'pitcher_level', 'pitcher_throws', 'date', 'pitch_type', 'pitch_number', 'release_speed', 'spin_rate', 'induced_vertical_break', 'horizontal_break', 'vertical_approach_angle', 'horizontal_approach_angle', 'vertical_break', 'spin_efficiency', 'spin_axis', 'tilt', 'extension', 'release_height', 'release_side', 'plate_location_height', 'plate_location_side']
